# A retrieval agent with BM25-style scoring

A complete, deterministic course companion.

## Learning objectives

- Tokenize a query and a small embedded corpus.
- Combine term frequency, document length, and inverse document frequency.
- Pass the top retrieved guest to an agent-facing result.

## Setup

The corpus and scoring code are embedded and use only `collections`, `math`, and `re`. Search credentials are deliberately outside the offline path.


For the optional live cell, install the dependency with `python -m pip install smolagents`.

## Complete implementation

Run the next cell to build the offline demonstration.

In [1]:
from collections import Counter
import math
import re

guest_corpus = [
    {"name": "Ada Lovelace", "bio": "mathematician first programmer analytical engine algorithm"},
    {"name": "Alan Turing", "bio": "mathematician computing cryptanalysis machine intelligence"},
    {"name": "Grace Hopper", "bio": "computer scientist compiler programming language navy"},
]

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.casefold())

documents = [tokenize(item["bio"]) for item in guest_corpus]
query_tokens = tokenize("first programmer analytical engine")
average_length = sum(map(len, documents)) / len(documents)
k1, b = 1.5, 0.75

def bm25_style_score(document):
    frequencies = Counter(document)
    score = 0.0
    for term in query_tokens:
        containing = sum(term in candidate for candidate in documents)
        inverse_frequency = math.log(1 + (len(documents) - containing + 0.5) / (containing + 0.5))
        term_frequency = frequencies[term]
        denominator = term_frequency + k1 * (1 - b + b * len(document) / average_length)
        score += inverse_frequency * (term_frequency * (k1 + 1) / denominator if term_frequency else 0)
    return score

ranked_guests = sorted(
    ((bm25_style_score(document), guest) for document, guest in zip(documents, guest_corpus)),
    key=lambda pair: (-pair[0], pair[1]["name"]),
)
top_score, top_record = ranked_guests[0]
top_guest = top_record["name"]

## Deterministic assertions

These checks make the lesson reproducible without a model or network connection.

In [2]:
assert query_tokens == ["first", "programmer", "analytical", "engine"]
assert top_guest == "Ada Lovelace"
assert top_score > ranked_guests[1][0]

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

Ada's record contains every query term, so it receives the highest score. The formula rewards relevant term matches while normalizing for document length; a production retriever would also evaluate recall, chunking, metadata filters, and answer grounding.

## Credentials

Offline retrieval needs no key. `SERPER_API_KEY` is for the Serper service; `SERPAPI_API_KEY` is for SerpAPI. They are different provider credentials for optional web-search extensions, and neither is used or required by this embedded retrieval tool. The live cell reads only `HF_TOKEN` and never prints its value.

In [3]:
import os

live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "huggingface").strip().casefold()
hf_token = os.getenv("HF_TOKEN")
if live_model_provider != "local" and not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional smolagents example.")
else:
    try:
        from smolagents import ToolCallingAgent, tool
        if live_model_provider == "local":
            from smolagents import TransformersModel
        else:
            from smolagents import InferenceClientModel
    except ImportError:
        print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
    else:
        if live_model_provider == "local":
            model = TransformersModel(
                model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                device_map="cpu",
                torch_dtype="auto",
                max_new_tokens=256,
            )
        else:
            model = InferenceClientModel(
                model_id="Qwen/Qwen2.5-Coder-32B-Instruct", token=hf_token, tool_choice="auto"
            )
        @tool
        def retrieve_guest(query: str) -> str:
            """Retrieve the most relevant guest from the embedded corpus.

            Args:
                query: Words describing the guest to find.
            """
            live_query_tokens = tokenize(query)

            def live_bm25_score(document):
                frequencies = Counter(document)
                score = 0.0
                for term in live_query_tokens:
                    containing = sum(term in candidate for candidate in documents)
                    inverse_frequency = math.log(
                        1 + (len(documents) - containing + 0.5) / (containing + 0.5)
                    )
                    term_frequency = frequencies[term]
                    denominator = term_frequency + k1 * (
                        1 - b + b * len(document) / average_length
                    )
                    score += inverse_frequency * (
                        term_frequency * (k1 + 1) / denominator if term_frequency else 0
                    )
                return score

            live_ranking = sorted(
                (
                    (live_bm25_score(document), guest)
                    for document, guest in zip(documents, guest_corpus)
                ),
                key=lambda pair: (-pair[0], pair[1]["name"]),
            )
            retrieved = live_ranking[0][1]
            return retrieved["name"] + ": " + retrieved["bio"]

        live_agent = ToolCallingAgent(tools=[retrieve_guest], model=model, max_steps=2)
        live_result = live_agent.run("Who best matches first programmer and analytical engine?")
        print(live_result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Who best matches first programmer and analytical engine?                                                        │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'search_agent' with arguments: {'query': 'First programmer and analytical engine', 'language':    │
│ 'english'}                                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Unknown tool search_agent, should be one of: retrieve_guest, final_answer.

... (4 outputs left out)

[Step 2: Duration 36.36 seconds| Input tokens: 2,302 | Output tokens: 76]

Reached max steps.

[Step 3: Duration 25.24 seconds| Input tokens: 2,569 | Output tokens: 192]

The first programmer and analytical engine that comes to mind is John von Neumann. He was a Hungarian-American mathematician, physicist, computer scientist, and polymath who made significant contributions to many fields including mathematics, physics, engineering, economics, and computing. Von Neumann is often referred to as the "father of modern computing" and is credited with creating the first stored-program computer, the ENIAC, in 1945. Additionally, he developed the field of artificial intelligence and is known for his work in logic, set theory, and combinatorics.


## Completion checklist

- [ ] Tokenized the embedded corpus.
- [ ] Ranked documents with BM25-style scoring.
- [ ] Verified the top guest and provider-key distinction.

## Attribution

Original compact retriever informed by the official [smolagents agentic RAG example](https://huggingface.co/docs/smolagents/examples/rag) and [agent reference](https://huggingface.co/docs/smolagents/reference/agents). The scoring implementation is newly authored for this lesson. Notebook code is released under the Apache-2.0 metadata license.